In [1]:
import os
from typing import Union

from torch.nn import CrossEntropyLoss

os.environ['http_proxy'] = 'http://127.0.0.1:7890'
os.environ['https_proxy'] = 'http://127.0.0.1:7890'
os.environ['all_proxy'] = 'socks5://127.0.0.1:7890'

os.environ['HF_HOME'] = '/home/bobsun/cambrige/MedMinist/hugginface'
os.environ['HUGGINGFACE_HUB_CACHE'] = '/home/bobsun/cambrige/MedMinist/hugginface'
from datasets import load_dataset, DatasetDict, Dataset, IterableDatasetDict, IterableDataset

# 加载 AG News 数据集
dataset = load_dataset('ag_news')

In [2]:
from transformers import BertTokenizer
# 加载 BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def preprocess_function(examples):
    tokenized = tokenizer(examples['text'], padding='max_length', truncation=True, max_length=71)
    return {
        'input_ids': tokenized['input_ids'],
        'attention_mask': tokenized['attention_mask'],
        'labels': examples['label']
    }

tokenized_datasets: Union[DatasetDict, Dataset, IterableDatasetDict, IterableDataset] = dataset.map(preprocess_function, batched=True, remove_columns=["text"])


In [3]:
from transformers import DataCollatorWithPadding
from torch.utils.data import DataLoader

train_dataset = tokenized_datasets["train"]
test_dataset = tokenized_datasets["test"]

# 定义 DataCollator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding='max_length', max_length=71, return_tensors='pt')

# 创建 DataLoader，指定 collate_fn
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=data_collator)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=data_collator)


In [4]:
import torch

# 假设 'pretrained_state_dict' 是预训练模型的 state_dict
pretrained_state_dict = torch.load('bert_ag_news.pth')  # 根据实际路径修改

new_state_dict = {}
for key, value in pretrained_state_dict.items():
    # 移除 'bert.' 前缀
    if key.startswith('bert.'):
        new_key = key[5:]  # 移除前5个字符
    else:
        new_key = key

    # 重命名分类头的权重和偏置
    if new_key == 'classifier.out_proj.weight':
        new_key = 'classifier.weight'
    elif new_key == 'classifier.out_proj.bias':
        new_key = 'classifier.bias'

    # 删除不需要的键
    keys_to_remove = [
        'encoder.layer.11.output.dense.weight',
        'encoder.layer.11.output.dense.bias',
        'encoder.layer.11.output.LayerNorm.weight',
        'encoder.layer.11.output.LayerNorm.bias',
        'pooler.dense.weight',
        'pooler.dense.bias',
        'classifier.weight',
        'classifier.bias'
    ]
    if new_key in keys_to_remove:
        continue  # 跳过这些键，不添加到 new_state_dict 中

    new_state_dict[new_key] = value


In [8]:
import importlib
import Transformer.DEN_BERT

importlib.reload(Transformer.DEN_BERT)  # 然后重新加载模块


# 初始化自定义模型
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

config = Transformer.DEN_BERT.BertConfig(
        num_labels=4  # 根据 AG News 数据集的标签数调整
    )
model = Transformer.DEN_BERT.CustomBertForSequenceClassification(config)

model_dict = model.state_dict()

# 用于记录匹配和不匹配的层名
matched_layers = []
unmatched_layers = []

# 筛选预训练的权重，只保留形状匹配的层
for k, v in new_state_dict.items():
    if k in model_dict and v.size() == model_dict[k].size():
        matched_layers.append(k)  # 记录匹配的层
    else:
        unmatched_layers.append(k)  # 记录不匹配的层

# 更新模型的 state_dict，只加载匹配的层
model_dict.update({k: v for k, v in new_state_dict.items() if k in matched_layers})
model.load_state_dict(model_dict)

# 打印报告
print("Matched Layers:")
for layer in matched_layers:
    print(f"  - {layer}")
print("\nUnmatched Layers:")
for layer in unmatched_layers:
    print(f"  - {layer}")

model.to(device)


Matched Layers:
  - embeddings.word_embeddings.weight
  - embeddings.position_embeddings.weight
  - embeddings.token_type_embeddings.weight
  - embeddings.LayerNorm.weight
  - embeddings.LayerNorm.bias
  - encoder.layer.0.attention.self.query.weight
  - encoder.layer.0.attention.self.query.bias
  - encoder.layer.0.attention.self.key.weight
  - encoder.layer.0.attention.self.key.bias
  - encoder.layer.0.attention.self.value.weight
  - encoder.layer.0.attention.self.value.bias
  - encoder.layer.0.attention.output.dense.weight
  - encoder.layer.0.attention.output.dense.bias
  - encoder.layer.0.attention.output.LayerNorm.weight
  - encoder.layer.0.attention.output.LayerNorm.bias
  - encoder.layer.0.intermediate.dense.weight
  - encoder.layer.0.intermediate.dense.bias
  - encoder.layer.1.attention.self.query.weight
  - encoder.layer.1.attention.self.query.bias
  - encoder.layer.1.attention.self.key.weight
  - encoder.layer.1.attention.self.key.bias
  - encoder.layer.1.attention.self.value.w

CustomBertForSequenceClassification(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropou

In [10]:
from transformers import BertForSequenceClassification, get_linear_schedule_with_warmup
import torch
from tqdm import tqdm
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss
# from transformers import logging
# import warnings
# warnings.filterwarnings("ignore", message="Some weights of BertForSequenceClassification were not initialized from the model checkpoint")
#
# # 加载预训练的 BERT 模型
#
# logging.set_verbosity_error()
# 如果有可用的 GPU，请将模型转移到 GPU 上
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

# 使用 AdamW 优化器
optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)

# 定义学习率调度器
epochs = 5
total_steps = len(train_dataloader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

# 定义损失函数（已在模型内部实现）
# 损失已经由 BertForSequenceClassification 计算
criterion = CrossEntropyLoss()

for epoch in range(epochs):
    model.train()  # 切换到训练模式
    total_loss = 0

    # 使用 tqdm 显示训练进度条
    for batch in tqdm(train_dataloader, desc=f"Training Epoch {epoch +1}"):
        # 获取输入数据和标签
        input_ids = batch['input_ids'][:, 1:].to(device)         # no [cls]
        attention_mask = batch['attention_mask'][:, 1:].to(device)  # no [cls]
        labels = batch['labels'].to(device)


        # 清除之前的梯度
        optimizer.zero_grad()

        # 前向传播
        outputs = model(input_ids, attention_mask=attention_mask)
        loss = 0
        if isinstance(outputs, list):
            for output in outputs:
                loss += criterion(output, labels)
        else: # 这个只是以防万一,一般来说就是list了
            print("not list")

        # 反向传播
        loss.backward()

        # 梯度裁剪
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # 优化器更新参数
        optimizer.step()

        # 更新学习率
        scheduler.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_dataloader)
    print(f"Epoch {epoch +1}, Average Training Loss: {avg_train_loss:.4f}")

    # 验证步骤
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in tqdm(test_dataloader, desc=f"Validation Epoch {epoch +1}"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            output = model(input_ids, attention_mask=attention_mask)

            logits = output

            predictions = torch.argmax(logits, dim=-1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    accuracy = correct / total
    print(f"Epoch {epoch +1}, Validation Accuracy: {accuracy * 100:.2f}%")


Training Epoch 1: 100%|██████████| 3750/3750 [03:16<00:00, 19.05it/s]


Epoch 1, Average Training Loss: 1.8417


Validation Epoch 1: 100%|██████████| 238/238 [00:03<00:00, 67.33it/s]


Epoch 1, Validation Accuracy: 93.32%


Training Epoch 2: 100%|██████████| 3750/3750 [03:16<00:00, 19.05it/s]


...

Epoch 3, Average Training Loss: 1.1830


Validation Epoch 3: 100%|██████████| 238/238 [00:03<00:00, 67.34it/s]


Epoch 3, Validation Accuracy: 93.47%


Training Epoch 4:  55%|█████▍    | 2050/3750 [01:47<01:29, 19.10it/s]


KeyboardInterrupt: 

In [11]:
torch.save(model.state_dict(), 'den_ag_news.pth')
